# Autoportrait dans le rap — Qwen3-32B

Annotation des **200 chansons de `chansons_test_200.csv`** avec trois décisions : `autoportrait`, `pas_autoportrait`, `je_ne_sais_pas`, et une justification en français, sans citation des paroles.

La définition est lue directement dans [consigne.md](consigne.md), sans modification. Le nettoyage, la construction des messages et le schéma JSON viennent de [annoter.py](annoter.py). Les annotations Claude ne sont **jamais envoyées au modèle**.

## Utilisation
1. Choisir un noyau Jupyter avec **pandas**, **tqdm** et **statsmodels** pour la comparaison (par exemple **Rap R05b (PyTorch CPU)**). Les dépendances sont listées dans [requirements.txt](requirements.txt) : `python -m pip install -r requirements.txt` depuis ce dossier, avec le Python du noyau. Le client ne charge aucun modèle sur GPU.
2. Vérifier les paramètres et disposer d'un serveur **vLLM / Qwen/Qwen3-32B**, par défaut `http://127.0.0.1:8016/v1`. Le carnet réutilise le serveur : il ne démarre ni n'arrête de processus et ne télécharge pas de modèle. Une autre adresse peut être renseignée via `QWEN_BASE_URL`.
3. Exécuter les cellules jusqu'au **pilote de 5 chansons**, puis examiner les décisions et justifications.
4. Après revue humaine, mettre `pilote_approuve = True` et `lancer_les_200 = True` dans les paramètres, puis réexécuter les cellules. Le pilote validé est repris depuis son journal ; le traitement des 200 est alors autorisé.

Les résultats vont dans `data/results/`, les métriques dans `data/performance/`, sous ce dossier. Les journaux JSONL sont sauvegardés après chaque réponse ; une relance reprend les réussites et retente les erreurs. Le pilote et les 200 ont des sorties séparées. **Une erreur réseau, HTTP ou JSON n'est jamais une décision `je_ne_sais_pas`.**

La température est fixée à 0 et le raisonnement Qwen3 est désactivé. Les paroles sont envoyées intégralement après nettoyage : aucune troncature silencieuse, aucun filtre supplémentaire sur l'échantillon fourni.

In [2]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter
from IPython.display import display
import csv
import fcntl
import hashlib
import importlib.util
import json
import os
import time
import urllib.error
import urllib.request

import pandas as pd
from tqdm.auto import tqdm

# Accepte un lancement depuis la racine du dépôt, Rap ou Autoportrait.
candidats = [Path.cwd(), Path.cwd() / 'Autoportrait', Path.cwd() / 'Rap' / 'Autoportrait']
dossiers = [p.resolve() for p in candidats if (p / 'consigne.md').is_file() and (p / 'annoter.py').is_file()]
if len(dossiers) != 1:
    raise RuntimeError('Lancer le carnet depuis la racine du dépôt, Rap ou Rap/Autoportrait.')
dossier = dossiers[0]
corpus_path = dossier.parent / 'corpus.csv'
echantillon_path = dossier / 'chansons_test_200.csv'
consigne_path = dossier / 'consigne.md'
resultats_dir = dossier / 'data' / 'results'
performance_dir = dossier / 'data' / 'performance'

base_url = os.environ.get('QWEN_BASE_URL', 'http://127.0.0.1:8016/v1').rstrip('/')
modele = 'Qwen/Qwen3-32B'
cle_api = os.environ.get('QWEN_API_KEY', 'EMPTY')
taille_pilote = 5
graine = 42
concurrence = 8
timeout_secondes = 180
max_tokens = 512
max_essais = 2
pilote_approuve = False
lancer_les_200 = False

if min(taille_pilote, concurrence, timeout_secondes, max_tokens, max_essais) <= 0:
    raise ValueError('Les tailles, délais et nombres de tentatives doivent être positifs.')
spec = importlib.util.spec_from_file_location('autoportrait_source', dossier / 'annoter.py')
if spec is None or spec.loader is None:
    raise ImportError('Impossible de charger annoter.py.')
source = importlib.util.module_from_spec(spec)
spec.loader.exec_module(source)
consigne = consigne_path.read_text(encoding='utf-8').strip()
if not consigne:
    raise ValueError('La consigne est vide.')
print('Dossier :', dossier)
print('Modèle :', modele, '| endpoint :', base_url)


Dossier : /home/cbenavent/test/Rap/Autoportrait
Modèle : Qwen/Qwen3-32B | endpoint : http://127.0.0.1:8016/v1


## 1. Retrouver les paroles et vérifier les correspondances

L'échantillon contient des métadonnées, pas les paroles. `source_row_index` désigne la **position de la ligne de données, à partir de zéro**, dans `../corpus.csv`, comme dans le script fourni. `document_id` est conservé comme métadonnée de l'échantillon ; il n'est pas utilisé comme position ni comme clé `doc_id` du corpus. Le titre et l'artiste sont contrôlés pour détecter un corpus réordonné. Les 200 lignes sont conservées dans leur ordre initial. Les colonnes d'annotation ajoutées à l'échantillon ne sont ni envoyées au modèle ni utilisées dans l'empreinte de reprise.

In [3]:
def charger_chansons():
    with echantillon_path.open(newline='', encoding='utf-8-sig') as fichier:
        champs = ('source_row_index', 'document_id', 'periode', 'artiste', 'titre', 'annee', 'mots')
        echantillon = [{champ: ligne[champ] for champ in champs} for ligne in csv.DictReader(fichier)]
    if len(echantillon) != 200:
        raise ValueError(f'200 chansons attendues, {len(echantillon)} trouvées.')
    ids = [int(ligne['source_row_index']) for ligne in echantillon]
    if len(set(ids)) != len(ids):
        raise ValueError('source_row_index contient des doublons.')
    corpus = source.lire_corpus(corpus_path, set(ids))
    manquants = sorted(set(ids) - set(corpus))
    if manquants:
        raise ValueError(f'Chansons absentes ou paroles vides : {manquants}')
    chansons = []
    for meta, position in zip(echantillon, ids):
        chanson = corpus[position]
        artiste = meta['artiste'].replace('\u200b', '').strip()
        if artiste != chanson['artiste'] or meta['titre'].strip() != chanson['titre'].strip():
            raise ValueError(f'Artiste/titre incohérents à la position {position} : vérifier corpus.csv.')
        paroles = source.nettoyer(chanson['paroles'])
        if not paroles:
            raise ValueError(f'Paroles vides après nettoyage : {position}')
        chansons.append({**meta, 'source_row_index': position, 'paroles': paroles,
                         'mots_nettoyes': len(paroles.split())})
    return chansons

chansons = charger_chansons()
display(pd.DataFrame(chansons).drop(columns='paroles').head())
print(f'{len(chansons)} chansons retrouvées ; aucune exclue ni tronquée.')


,source_row_index,document_id,periode,artiste,titre,annee,mots,mots_nettoyes
0,29159,28826,1990-1999,MC Solaar,Wonderbra,1997,467,467
1,24039,23752,1990-1999,IAM,Le Feu,1993,654,654
2,5419,5390,1990-1999,Faf Larage,A cet instant précis,1999,403,403
3,28549,28221,1990-1999,Stomy Bugsy,Le Playboy de Sarcelles,1997,682,682
4,28551,28223,1990-1999,Passi,Les meufs du show-biz,1997,760,760


200 chansons retrouvées ; aucune exclue ni tronquée.


## 2. Client vLLM et validation stricte

Le serveur doit annoncer exactement le modèle configuré. Les réponses sont contraintes par le schéma JSON de `annoter.py`, puis vérifiées côté client (clés exactes, justification non vide, étiquette autorisée, génération terminée). Les erreurs transitoires sont retentées ; les erreurs HTTP permanentes, dont un dépassement de contexte, sont conservées explicitement. Aucun basculement silencieux vers un autre modèle ou vers du texte libre.

In [4]:
ouvreur = urllib.request.build_opener(urllib.request.ProxyHandler({}))

def appeler_api(route, corps=None):
    requete = urllib.request.Request(
        base_url + route,
        data=None if corps is None else json.dumps(corps).encode('utf-8'),
        headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {cle_api}'},
    )
    with ouvreur.open(requete, timeout=timeout_secondes) as reponse:
        return json.loads(reponse.read().decode('utf-8'))

def verifier_serveur():
    modeles = [item['id'] for item in appeler_api('/models')['data']]
    if modele not in modeles:
        raise RuntimeError(f'{modele} non servi. Modèles disponibles : {modeles}')
    print('Serveur disponible :', modele)

def valider_reponse(reponse):
    choix = reponse['choices'][0]
    if choix.get('finish_reason') != 'stop':
        raise ValueError(f"Génération incomplète : {choix.get('finish_reason')!r}")
    contenu = choix['message']['content']
    if not isinstance(contenu, str):
        raise ValueError('Contenu absent ou non textuel.')
    objet = json.loads(contenu)
    if not isinstance(objet, dict) or set(objet) != {'justification', 'etiquette'}:
        raise ValueError('Deux clés exactes attendues : justification et etiquette.')
    if objet['etiquette'] not in source.ETIQUETTES:
        raise ValueError(f"Étiquette inconnue : {objet['etiquette']!r}")
    if not isinstance(objet['justification'], str) or not objet['justification'].strip():
        raise ValueError('Justification vide ou non textuelle.')
    return {'justification': objet['justification'].strip(), 'etiquette': objet['etiquette']}

def annoter_chanson(chanson):
    debut = time.perf_counter()
    corps = {
        'model': modele,
        'messages': source.messages(consigne, chanson['paroles'], False),
        'temperature': 0, 'seed': graine, 'max_tokens': max_tokens,
        'chat_template_kwargs': {'enable_thinking': False},
        **source.CONTRAINTES['json_schema'],
    }
    for essai in range(1, max_essais + 1):
        try:
            reponse = appeler_api('/chat/completions', corps)
            annotation = valider_reponse(reponse)
            return {**annotation, 'statut': 'ok', 'erreur': '', 'essais': essai,
                    'secondes': time.perf_counter() - debut}
        except urllib.error.HTTPError as exc:
            erreur = f'HTTP {exc.code} : {exc.read().decode("utf-8", "replace")[:1000]}'
            if exc.code not in {408, 429, 500, 502, 503, 504}:
                break
        except (OSError, ValueError, KeyError, IndexError, TypeError) as exc:
            erreur = f'{type(exc).__name__} : {exc}'
        if essai < max_essais:
            time.sleep(2 * essai)
    return {'justification': '', 'etiquette': '', 'statut': 'erreur', 'erreur': erreur,
            'essais': essai, 'secondes': time.perf_counter() - debut}


## 3. Journal, reprise, progression et exports

Une empreinte identifie le modèle, l'URL, la consigne, le code de nettoyage, les paramètres et les textes sélectionnés. Un changement crée donc un nouveau journal. Les reprises ne mélangent pas des consignes ou des corpus différents. Un verrou empêche deux lancements simultanés du même périmètre. Une interruption conserve les réponses déjà écrites ; les requêtes en cours peuvent terminer avant que la cellule rende la main.

La barre affiche le débit et l'ETA **mesurés côté client**. Les métriques ne prétendent pas mesurer la mémoire, le temps GPU ou le carbone d'un serveur distant. Les exports contiennent une ligne par chanson, erreurs comprises, dans l'ordre de l'échantillon. Les taux par étiquette excluent les erreurs techniques.

In [5]:
def empreinte(selection):
    configuration = {
        'version_pipeline': 1, 'modele': modele, 'url': base_url, 'consigne': consigne,
        'annoter_sha256': hashlib.sha256((dossier / 'annoter.py').read_bytes()).hexdigest(),
        'temperature': 0, 'seed': graine, 'max_tokens': max_tokens,
        'enable_thinking': False, 'contrainte': source.CONTRAINTES['json_schema'],
        'selection': selection,
    }
    return hashlib.sha256(json.dumps(configuration, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def executer_annotation(selection, perimetre):
    if not selection:
        raise ValueError('Sélection vide.')
    verifier_serveur()
    resultats_dir.mkdir(parents=True, exist_ok=True)
    performance_dir.mkdir(parents=True, exist_ok=True)
    signature = empreinte(selection)
    prefixe = f'autoportrait_qwen32_{perimetre}_n{len(selection)}_{signature[:12]}'
    journal = resultats_dir / f'{prefixe}.jsonl'
    ids = [c['source_row_index'] for c in selection]
    if len(ids) != len(set(ids)):
        raise ValueError('Sélection avec doublons.')
    debut = time.perf_counter()
    with (resultats_dir / f'{prefixe}.lock').open('a') as verrou:
        try:
            fcntl.flock(verrou, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError('Une annotation identique est déjà en cours.') from exc
        resultats = {}
        if journal.exists():
            with journal.open(encoding='utf-8') as fichier:
                for numero, ligne in enumerate(fichier, 1):
                    try:
                        resultat = json.loads(ligne)
                    except json.JSONDecodeError as exc:
                        raise ValueError(f'Journal endommagé : {journal}, ligne {numero}. Réparer avant reprise.') from exc
                    if resultat['empreinte'] != signature or resultat['source_row_index'] not in ids:
                        raise ValueError('Journal incompatible avec la sélection courante.')
                    if resultat['statut'] not in {'ok', 'erreur'}:
                        raise ValueError('Statut invalide dans le journal.')
                    if resultat['statut'] == 'ok':
                        valider_reponse({'choices': [{'finish_reason': 'stop', 'message': {'content':
                            json.dumps({k: resultat[k] for k in ('justification', 'etiquette')})}}]})
                    resultats[resultat['source_row_index']] = resultat
        restantes = [c for c in selection if resultats.get(c['source_row_index'], {}).get('statut') != 'ok']
        reprises = len(selection) - len(restantes)
        print(f'{perimetre} : {reprises} réussites reprises ; {len(restantes)} chansons à traiter.')
        traitees = erreurs = 0
        with journal.open('a', encoding='utf-8') as fichier:
            with ThreadPoolExecutor(max_workers=concurrence) as executeur:
                taches = {executeur.submit(annoter_chanson, c): c for c in restantes}
                try:
                    with tqdm(total=len(selection), initial=reprises, desc=f'Annotation {perimetre}',
                              unit='chansons', mininterval=0.5) as barre:
                        for tache in as_completed(taches):
                            chanson = taches[tache]
                            resultat = {**{k: v for k, v in chanson.items() if k != 'paroles'},
                                        **tache.result(), 'modele': modele, 'empreinte': signature,
                                        'date_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
                            fichier.write(json.dumps(resultat, ensure_ascii=False) + '\n')
                            fichier.flush()
                            os.fsync(fichier.fileno())
                            resultats[chanson['source_row_index']] = resultat
                            traitees += 1
                            erreurs += resultat['statut'] == 'erreur'
                            barre.update(1)
                            barre.set_postfix(erreurs=erreurs,
                                chansons_s=f'{traitees / max(time.perf_counter() - debut, 1e-9):.2f}', refresh=False)
                except BaseException:
                    for tache in taches:
                        tache.cancel()
                    raise
        ordonnes = [resultats[position] for position in ids]
        if len(ordonnes) != len(selection):
            raise RuntimeError('Effectif final incohérent.')
        tableau = pd.DataFrame(ordonnes)
        tableau.to_csv(resultats_dir / f'{prefixe}.csv', index=False, encoding='utf-8-sig')
        (resultats_dir / f'{prefixe}.json').write_text(
            json.dumps(ordonnes, ensure_ascii=False, indent=2), encoding='utf-8')
        tableau.loc[tableau['statut'] == 'erreur'].to_csv(
            resultats_dir / f'{prefixe}_erreurs.csv', index=False, encoding='utf-8-sig')
        secondes = time.perf_counter() - debut
        metriques = {'perimetre': perimetre, 'modele': modele, 'empreinte': signature,
                     'selection': len(selection), 'reprises': reprises, 'traitees_session': traitees,
                     'reussites': int((tableau['statut'] == 'ok').sum()),
                     'erreurs': int((tableau['statut'] == 'erreur').sum()),
                     'secondes_session_client': secondes, 'chansons_s_session': traitees / secondes,
                     'concurrence': concurrence, 'max_tokens': max_tokens}
        pd.DataFrame([metriques]).to_csv(performance_dir / f'{prefixe}_performance.csv', index=False)
        valides = tableau.loc[tableau['statut'] == 'ok']
        comptes = Counter(valides['etiquette'])
        distribution = pd.DataFrame([
            {'etiquette': e, 'effectif': comptes[e],
             'pourcentage_valides': 100 * comptes[e] / len(valides) if len(valides) else None}
            for e in source.ETIQUETTES])
        distribution.to_csv(performance_dir / f'{prefixe}_distribution.csv', index=False)
    display(pd.DataFrame([metriques]))
    display(distribution)
    if metriques['erreurs']:
        print('ATTENTION : erreurs techniques présentes ; voir le CSV erreurs et relancer pour les retenter.')
    print('Annotations :', resultats_dir / f'{prefixe}.csv')
    return tableau


## 4. Pilote reproductible et revue humaine

Le contrôle `/v1/models` précède les premières requêtes. Ce pilote valide aussi l'acceptation du schéma et la désactivation du raisonnement par le serveur. Examiner si les justifications appliquent bien **AUTO + PORTRAIT**, plutôt que la seule présence de « je ». Une sortie JSON valide n'est pas une garantie de justesse sémantique. Aucun traitement des 200 n'est lancé automatiquement.

In [6]:
if taille_pilote > len(chansons):
    raise ValueError('Le pilote dépasse la taille de l’échantillon.')
indices_pilote = sorted(pd.Series(range(len(chansons))).sample(n=taille_pilote, random_state=graine).tolist())
selection_pilote = [chansons[i] for i in indices_pilote]
resultats_pilote = executer_annotation(selection_pilote, 'pilote')
pilote_techniquement_valide = (len(resultats_pilote) == taille_pilote
                                and resultats_pilote['statut'].eq('ok').all())
display(resultats_pilote[['source_row_index', 'artiste', 'titre', 'etiquette', 'justification', 'erreur']])
if not pilote_techniquement_valide:
    raise RuntimeError('Pilote invalide : corriger les erreurs avant de traiter les 200 chansons.')
print('Pilote techniquement valide. Revue humaine nécessaire avant approbation.')


Serveur disponible : Qwen/Qwen3-32B
pilote : 5 réussites reprises ; 0 chansons à traiter.


Annotation pilote: 100%|##########| 5/5 [00:00<?, ?chansons/s]

,perimetre,modele,empreinte,selection,reprises,traitees_session,reussites,erreurs,secondes_session_client,chansons_s_session,concurrence,max_tokens
0,pilote,Qwen/Qwen3-32B,84d71fc9af107b970994b71e75f7545b1a06f9ccc63696...,5,5,0,5,0,0.012617,0.0,8,512


,etiquette,effectif,pourcentage_valides
0,autoportrait,3,60.0
1,je_ne_sais_pas,0,0.0
2,pas_autoportrait,2,40.0


Annotations : /home/cbenavent/test/Rap/Autoportrait/data/results/autoportrait_qwen32_pilote_n5_84d71fc9af10.csv


,source_row_index,artiste,titre,etiquette,justification,erreur
0,26490,MC Solaar,Dieu ait son âme,pas_autoportrait,Le texte parle surtout de personnes décédées e...,
1,15626,Svinkels,Boule puante,pas_autoportrait,Le texte ne présente pas celui qui parle comme...,
2,9311,Alpha 5.20,Scarface d’Afrique,autoportrait,"Le texte présente surtout celui qui parle, en ...",
3,25979,Odor,100 balles,autoportrait,"Le texte parle surtout de celui qui parle, en ...",
4,65647,S-Tee (FRA),Remember,autoportrait,"Le texte se concentre sur le rappeur lui-même,...",


Pilote techniquement valide. Revue humaine nécessaire avant approbation.


## 5. Annoter les 200 chansons après approbation

Activer les deux paramètres en tête du carnet après inspection du pilote, puis réexécuter. Ce périmètre possède son propre journal : les 5 chansons pilotes y seront réannotées pour garder des sorties indépendantes. Les erreurs éventuelles apparaissent dans le CSV dédié et sont retentées à la prochaine exécution. Le carnet ne calcule pas une « précision » sans protocole d'évaluation humaine.

In [7]:
if lancer_les_200:
    if not pilote_approuve:
        raise RuntimeError('Approuver explicitement le pilote après revue des justifications.')
    if not pilote_techniquement_valide or empreinte(selection_pilote) != resultats_pilote['empreinte'].iloc[0]:
        raise RuntimeError('Réexécuter le pilote avec les paramètres actuels avant de continuer.')
    resultats_200 = executer_annotation(chansons, 'test200')
    display(resultats_200[['source_row_index', 'artiste', 'titre', 'etiquette', 'justification', 'erreur']])
else:
    print('Traitement des 200 désactivé : examiner le pilote puis activer les deux paramètres d’approbation.')


Traitement des 200 désactivé : examiner le pilote puis activer les deux paramètres d’approbation.


## 6. Comparaison Qwen3-32B / Claude sur les 200 chansons

**Sans relancer l'annotation :** exécuter uniquement la cellule des paramètres puis les cellules de cette section. Elles relisent `chansons_test_200.csv` (colonnes `annotation_qwen32` et `justification_qwen32`) et `reference_claude_200.csv`. Appariement strict un-à-un par `source_row_index`, contrôle des 200 identifiants, titres et artistes ; les valeurs manquantes ou inconnues provoquent une erreur explicite.

Les trois étiquettes sont **nominales**, y compris `je_ne_sais_pas` : kappa de Cohen **non pondéré**, sans exclure les hésitations. Les tableaux ont **Claude en lignes et Qwen en colonnes** ; les pourcentages sont calculés par ligne et par colonne, sans compter les marges dans les dénominateurs.

Le kappa corrige l'accord observé par l'accord attendu à partir des marges. Le test bilatéral asymptotique porte sur **H₀ : κ = 0**, avec erreur-type sous H₀ ; l'IC à 95 % utilise l'erreur-type estimée sous l'alternative (Wald, non tronqué). Ce test n'évalue ni l'égalité des distributions marginales ni la qualité des annotations. Claude est un comparateur, **pas une vérité terrain**. Les résultats supposent des chansons indépendantes ; des chansons d'un même artiste peuvent rendre l'incertitude asymptotique trop optimiste.

Exports : appariements et désaccords dans `data/results/` ; contingences, distributions marginales et statistiques dans `data/performance/`. Les fichiers de comparaison sont remplacés lors d'une nouvelle exécution, sans toucher aux annotations sources.

In [8]:
from statsmodels.stats.inter_rater import cohens_kappa

categories_comparaison = ['autoportrait', 'pas_autoportrait', 'je_ne_sais_pas']
reference_claude_path = dossier / 'reference_claude_200.csv'
prefixe_comparaison = 'autoportrait_qwen32_claude_test200'

def apparier_annotations(qwen, claude):
    champs_qwen = ['source_row_index', 'artiste', 'titre', 'annotation_qwen32', 'justification_qwen32']
    champs_claude = ['source_row_index', 'artiste', 'titre', 'etiquette', 'justification']
    for nom, tableau, champs, etiquette in (
        ('Qwen', qwen, champs_qwen, 'annotation_qwen32'),
        ('Claude', claude, champs_claude, 'etiquette'),
    ):
        manquants = set(champs) - set(tableau.columns)
        if manquants:
            raise ValueError(f'{nom} : colonnes absentes : {sorted(manquants)}')
        if len(tableau) != 200 or tableau['source_row_index'].duplicated().any():
            raise ValueError(f'{nom} : 200 identifiants uniques sont requis.')
        if tableau['source_row_index'].eq('').any():
            raise ValueError(f'{nom} : identifiant vide.')
        invalides = ~tableau[etiquette].isin(categories_comparaison)
        if invalides.any():
            raise ValueError(f'{nom} : étiquettes invalides : {tableau.loc[invalides, etiquette].unique().tolist()}')
    if set(qwen['source_row_index']) != set(claude['source_row_index']):
        raise ValueError('Les deux fichiers ne portent pas sur les mêmes chansons.')
    reference = claude[champs_claude].rename(columns={
        'etiquette': 'annotation_claude', 'justification': 'justification_claude',
        'artiste': 'artiste_claude', 'titre': 'titre_claude'})
    comparaison = qwen[champs_qwen].merge(reference, on='source_row_index', how='left',
                                         sort=False, validate='one_to_one')
    for champ in ('artiste', 'titre'):
        gauche = comparaison[champ].str.replace('\u200b', '', regex=False).str.strip()
        droite = comparaison[f'{champ}_claude'].str.replace('\u200b', '', regex=False).str.strip()
        if not gauche.eq(droite).all():
            raise ValueError(f'Métadonnées incompatibles : {champ}.')
    comparaison = comparaison.drop(columns=['artiste_claude', 'titre_claude'])
    comparaison['accord'] = comparaison['annotation_qwen32'].eq(comparaison['annotation_claude'])
    return comparaison

def calculer_accord(comparaison):
    contingence = pd.crosstab(comparaison['annotation_claude'], comparaison['annotation_qwen32']).reindex(
        index=categories_comparaison, columns=categories_comparaison, fill_value=0)
    contingence.index.name = 'Claude'
    contingence.columns.name = 'Qwen3-32B'
    n = int(contingence.to_numpy().sum())
    if n == 0:
        raise ValueError('Aucune paire à comparer.')
    lignes = contingence.sum(axis=1)
    colonnes = contingence.sum(axis=0)
    accord_observe = float(contingence.to_numpy().trace() / n)
    accord_attendu = float((lignes * colonnes).sum() / n ** 2)
    if accord_attendu >= 1:
        raise ValueError('Kappa indéfini : les deux annotateurs utilisent une seule catégorie identique.')
    kappa = cohens_kappa(contingence.to_numpy())
    if not (kappa.std_kappa0 > 0):
        raise ValueError('Test asymptotique indéfini : variance nulle sous H0.')
    statistiques = pd.DataFrame([{
        'n': n, 'accords': int(comparaison['accord'].sum()),
        'desaccords': int((~comparaison['accord']).sum()),
        'accord_observe': accord_observe, 'accord_attendu': accord_attendu,
        'kappa_cohen_non_pondere': kappa.kappa, 'erreur_type_kappa': kappa.std_kappa,
        'ic95_bas_wald': kappa.kappa_low, 'ic95_haut_wald': kappa.kappa_upp,
        'erreur_type_sous_H0': kappa.std_kappa0, 'z_H0_kappa_0': kappa.z_value,
        'p_bilaterale_H0_kappa_0': kappa.pvalue_two_sided,
    }])
    return contingence, statistiques


In [9]:
qwen_compare = pd.read_csv(echantillon_path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
claude_compare = pd.read_csv(reference_claude_path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
comparaison = apparier_annotations(qwen_compare, claude_compare)
contingence, statistiques_accord = calculer_accord(comparaison)
contingence_marges = contingence.copy()
contingence_marges['Total'] = contingence.sum(axis=1)
contingence_marges.loc['Total'] = contingence_marges.sum(axis=0)
# Une catégorie absente donne une ligne/colonne non définie (NaN), pas un faux 0 %.
pourcentages_lignes = contingence.div(contingence.sum(axis=1).replace(0, float('nan')), axis=0) * 100
pourcentages_colonnes = contingence.div(contingence.sum(axis=0).replace(0, float('nan')), axis=1) * 100
marges = pd.DataFrame({'Claude_n': contingence.sum(axis=1), 'Qwen_n': contingence.sum(axis=0)})
marges['Claude_pct'] = marges['Claude_n'] / len(comparaison) * 100
marges['Qwen_pct'] = marges['Qwen_n'] / len(comparaison) * 100
desaccords = comparaison.loc[~comparaison['accord']].copy()

resultats_dir.mkdir(parents=True, exist_ok=True)
performance_dir.mkdir(parents=True, exist_ok=True)
comparaison.to_csv(resultats_dir / f'{prefixe_comparaison}_appariements.csv', index=False, encoding='utf-8-sig')
desaccords.to_csv(resultats_dir / f'{prefixe_comparaison}_desaccords.csv', index=False, encoding='utf-8-sig')
for nom, tableau in (
    ('contingence_effectifs', contingence_marges), ('contingence_pct_lignes', pourcentages_lignes),
    ('contingence_pct_colonnes', pourcentages_colonnes), ('marges', marges),
):
    tableau.to_csv(performance_dir / f'{prefixe_comparaison}_{nom}.csv', encoding='utf-8-sig')
statistiques_accord.to_csv(performance_dir / f'{prefixe_comparaison}_kappa.csv', index=False)

print('Contingence : Claude en lignes, Qwen3-32B en colonnes (effectifs et marges)')
display(contingence_marges)
print('Pourcentages par ligne Claude')
display(pourcentages_lignes.round(2))
print('Pourcentages par colonne Qwen')
display(pourcentages_colonnes.round(2))
print('Distributions marginales')
display(marges)
print('Accord et test bilatéral H0 : kappa = 0')
display(statistiques_accord)
bilan = statistiques_accord.iloc[0]
print(f"Accord : {bilan['accord_observe']:.1%} ; κ = {bilan['kappa_cohen_non_pondere']:.3f} ; "
      f"IC 95 % [{bilan['ic95_bas_wald']:.3f}, {bilan['ic95_haut_wald']:.3f}] ; "
      f"p bilatérale = {bilan['p_bilaterale_H0_kappa_0']:.3g}.")
print(f'{len(desaccords)} désaccords exportés, avec les deux justifications ; aperçu :')
display(desaccords.head(20))


Contingence : Claude en lignes, Qwen3-32B en colonnes (effectifs et marges)


Qwen3-32B,autoportrait,pas_autoportrait,je_ne_sais_pas,Total
Claude,,,,
autoportrait,107,14,1,122
pas_autoportrait,3,50,0,53
je_ne_sais_pas,6,18,1,25
Total,116,82,2,200


Pourcentages par ligne Claude


Qwen3-32B,autoportrait,pas_autoportrait,je_ne_sais_pas
Claude,,,
autoportrait,87.70,11.48,0.82
pas_autoportrait,5.66,94.34,0.00
je_ne_sais_pas,24.00,72.00,4.00


Pourcentages par colonne Qwen


...

,Claude_n,Qwen_n,Claude_pct,Qwen_pct
autoportrait,122,116,61.0,58.0
pas_autoportrait,53,82,26.5,41.0
je_ne_sais_pas,25,2,12.5,1.0


Accord et test bilatéral H0 : kappa = 0


,n,accords,desaccords,accord_observe,accord_attendu,kappa_cohen_non_pondere,erreur_type_kappa,ic95_bas_wald,ic95_haut_wald,erreur_type_sous_H0,z_H0_kappa_0,p_bilaterale_H0_kappa_0
0,200,158,42,0.79,0.4637,0.608428,0.046319,0.517645,0.699211,0.056586,10.752295,5.780500e-27


Accord : 79.0% ; κ = 0.608 ; IC 95 % [0.518, 0.699] ; p bilatérale = 5.78e-27.
42 désaccords exportés, avec les deux justifications ; aperçu :


,source_row_index,artiste,titre,annotation_qwen32,justification_qwen32,annotation_claude,justification_claude,accord
1,24039,IAM,Le Feu,pas_autoportrait,Le texte raconte surtout l'expérience d'un con...,je_ne_sais_pas,"Le groupe est bien le sujet, mais le texte rac...",False
11,40389,Manu Key,Outro,autoportrait,Le texte parle principalement de celui qui par...,pas_autoportrait,"Courte dédicace de fin d'album, adressée aux p...",False
16,28253,IAM,Achevez-moi,autoportrait,"Le texte exprime surtout l'état d'esprit, les ...",je_ne_sais_pas,Bref autoportrait d'un être dépressif et morbi...,False
20,27723,Fabe,Plus il y a de monde,pas_autoportrait,Le texte parle surtout de l'isolement ressenti...,autoportrait,"Le rappeur se peint seul dans la foule, étrang...",False
22,9366,Assassin (FRA),Où Va Se Diriger Ton Stylo ?,pas_autoportrait,Le texte ne se concentre pas principalement su...,je_ne_sais_pas,"Le rappeur dit quel artiste il refuse d'être, ...",False
26,2985,Ideal J,R.A.S. 1,pas_autoportrait,Le texte ne se concentre pas sur la descriptio...,autoportrait,Les rappeurs se définissent par leur refus du ...,False
30,15626,Svinkels,Boule puante,pas_autoportrait,Le texte ne présente pas celui qui parle comme...,autoportrait,Le groupe se présente sur le mode de la blague...,False
39,68975,Les Sages Poètes de la Rue,Ne Cours Pas,autoportrait,"Le texte se concentre sur celui qui parle, en ...",je_ne_sais_pas,L'egotrip des rappeurs alterne avec une leçon ...,False
40,2212,Unité 2 Feu,Père Noël,pas_autoportrait,Le texte ne se concentre pas sur celui qui par...,je_ne_sais_pas,Charge contre le Père Noël mêlée au portrait d...,False
43,9307,Svinkels,C’est Des Cons,pas_autoportrait,Le texte ne se concentre pas sur celui qui par...,je_ne_sais_pas,Satire du public du groupe ; seul le dernier c...,False
